# Task 8: throughput

Energy evaluations per second per model, on the same 4000 dataset positions. Baseline and student are the same code (`energy_fast`) with different parameter arrays, hence equally fast. The MACE time comes from task 6 (joltik, V100, all-atom CH4 in the cell of 424 atoms, 0.19 s per point).

In [1]:
import sys
sys.path.insert(0, "../src")
import time
import numpy as np
from energy import load_framework, load_force_field
from gcmc import CH4_EPS, CH4_SIGMA
from fit import element_indices, baseline_params, params_to_pairs, student_energies

positions, symbols, L, masses = load_framework("../data/IRMOF-1.cif")
lj_params = load_force_field("../data/force_field/generic_dreiding_uff_mixing_rules.def")
element_index = element_indices(symbols)
params_baseline = baseline_params(lj_params, (CH4_EPS, CH4_SIGMA))
params_student = np.loadtxt("../results/student_params.csv", delimiter=",", comments="#")[:, 2]   # column 'weighted'

data = np.genfromtxt("../data/mace_dataset_irmof1_ch4.csv", delimiter=",", names=True)
dataset_positions = np.column_stack([data["x"], data["y"], data["z"]])
N = len(dataset_positions)

/Users/kasperhuysentruyt/Documents/leven /Cusp/cusp-mlff-gcmc/.venv/lib/python3.14/site-packages/ase/io/cif.py:411: UserWarning: crystal system 'cubic' is not interpreted for space group Spacegroup(225, setting=1). This may result in wrong setting!
  warnings.warn(


In [2]:
def time_model(params, repeats=5):
    times = []
    for _ in range(repeats):
        t0 = time.time()
        student_energies(params, dataset_positions, positions, element_index, L)
        times.append(time.time() - t0)
    return np.median(times) / N   # s per evaluation, median over repeats

t_baseline = time_model(params_baseline)
t_student = time_model(params_student)
t_mace = 0.188   # s per point, task 6, job 40881994 on joltik (V100), mace_mp medium + D3, float64

rows = [("baseline DREIDING/UFF, energy_fast", "CPU (Apple M-series), united atom", t_baseline),
        ("student LJ, energy_fast", "CPU (Apple M-series), united atom", t_student),
        ("MACE-MP-0 medium + D3", "GPU (V100), all-atom CH4 + 424 framework atoms", t_mace)]
print(f"{'model':36s} {'hardware':46s} {'s / evaluation':>14s} {'evaluations / s':>15s} {'vs MACE':>12s}")
for name, hw, t in rows:
    print(f"{name:36s} {hw:46s} {t:14.2e} {1 / t:15.0f} {t_mace / t:12.0f}x")

print("\nMarkdown for the README:")
print("| model | hardware | s / evaluation | evaluations / s | faster than MACE |")
print("|---|---|---|---|---|")
for name, hw, t in rows:
    print(f"| {name} | {hw} | {t:.1e} | {1 / t:,.0f} | {t_mace / t:,.0f}x |")

model                                hardware                                       s / evaluation evaluations / s      vs MACE
baseline DREIDING/UFF, energy_fast   CPU (Apple M-series), united atom                    3.52e-05           28434         5346x
student LJ, energy_fast              CPU (Apple M-series), united atom                    3.53e-05           28341         5328x
MACE-MP-0 medium + D3                GPU (V100), all-atom CH4 + 424 framework atoms       1.88e-01               5            1x

Markdown for the README:
| model | hardware | s / evaluation | evaluations / s | faster than MACE |
|---|---|---|---|---|
| baseline DREIDING/UFF, energy_fast | CPU (Apple M-series), united atom | 3.5e-05 | 28,434 | 5,346x |
| student LJ, energy_fast | CPU (Apple M-series), united atom | 3.5e-05 | 28,341 | 5,328x |
| MACE-MP-0 medium + D3 | GPU (V100), all-atom CH4 + 424 framework atoms | 1.9e-01 | 5 | 1x |


## Caveats

- Apples and oranges, deliberately: an LJ sum on one CPU core versus an equivariant network on a GPU, and one united-atom point versus 429 atoms. That is exactly the comparison that matters, because in GCMC the energy of one trial molecule is the inner loop.
- A GCMC isotherm from task 4 cost ~120,000 steps at 1 to 2 evaluations each per pressure point. With the student that is seconds per pressure point; with MACE one pressure point would cost ~6 to 12 GPU-hours, and the whole seven-point isotherm several days.
- Another factor 10 to 100 is available by tabulating the framework energy on a 3D grid in advance and interpolating trilinearly (Frenkel & Smit appendix I, as RASPA does). Not done; listed in the README under next steps.